<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB07_Machine_Learning_Fundamentals_First_Real_Ship_Dataset_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB07 · Clase 7 — Fundamentos de Machine Learning con un dataset real de buques**

## Bloque 2: IA — Machine Learning (continuación)

Esta clase presenta el vocabulario y el flujo de trabajo básicos del Machine Learning (aprendizaje supervisado, conjuntos de entrenamiento/validación/test, sobreajuste, métricas de evaluación) y después lo aplica todo, de principio a fin, a un **dataset real y publicado**: el dataset [*Ship Fuel Consumption and CO2 Emissions Analysis*](https://www.kaggle.com/datasets/jeleeladekunlefijabi/ship-fuel-consumption-and-co2-emissions-analysis) de Kaggle, que cubre 120 buques operando en vías navegables de Nigeria durante 12 meses (1.440 registros de viaje). Está reflejado en este repositorio en [`Datasets/ship_fuel_efficiency.csv`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/ship_fuel_efficiency.csv), así que no hace falta clave de API ni paso de descarga — el mismo patrón que `Naval_Dataset.csv` en `NB02`.

Vamos a entrenar y evaluar **dos modelos reales** sobre estos datos:
- Un modelo de **clasificación** que predice qué tipo de combustible (Diesel o HFO) usó un viaje.
- Un modelo de **regresión** que predice el consumo de combustible a partir de las condiciones del viaje.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar el aprendizaje supervisado, no supervisado y por refuerzo, y por qué los datos se dividen en conjuntos de entrenamiento/validación/test.
- Definir el sobreajuste, el infraajuste y el equilibrio sesgo-varianza.
- Calcular e interpretar métricas de clasificación (accuracy, precisión, recall, F1, matriz de confusión) y métricas de regresión (MAE, MSE, RMSE, R²).
- Reconocer y evitar una fuga de datos (data leakage) usando un ejemplo real.
- Cargar un dataset real, construir características (one-hot encoding, escalado), y entrenar/evaluar un modelo de clasificación y otro de regresión.
- Usar validación cruzada k-fold para obtener una estimación del rendimiento más robusta que una única división train/test.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso de NB01–NB06, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Tipos de Machine Learning; conceptos clave (train/val/test, sobreajuste, sesgo-varianza) | 15 min | Teoría |
| 3 | Métricas de evaluación: clasificación (ejemplo resuelto) y regresión (fórmulas) | 15 min | Teoría |
| 4 | Carga y exploración del dataset real (`ship_fuel_efficiency.csv`) | 15 min | Práctica |
| 5 | Ingeniería de características y preprocesado, evitando la fuga de datos | 15 min | Teoría + Práctica |
| 6 | Clasificación en la práctica: predecir el tipo de combustible | 15 min | Práctica |
| 7 | Regresión en la práctica: predecir el consumo de combustible (Linear Regression vs. Random Forest) | 15 min | Práctica |
| 8 | Validación cruzada sobre el modelo real | 15 min | Teoría + Práctica |
| 9 | Aplicaciones prácticas del ML en ingeniería naval/oceánica (panorama general) | 5 min | Teoría |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son orientación aproximada, no un guion estricto — no hay descansos programados. Si terminamos todo con tiempo de sobra, la clase acaba antes; eso puede pasar y no es un problema.

---

## 1. Repaso: dónde estamos

- **`NB01`**: historia de la IA, IA débil vs. fuerte, por qué importa la IA para la ingeniería naval/oceánica.
- **`NB02`**: fundamentos de Python y Colab, arrays de NumPy, y nuestra primera mirada a un dataset naval (`Naval_Dataset.csv`) con Pandas — `head()`, `describe()`, filtrado, `groupby()`, `.corr()`, y gráficos sencillos.
- **`NB03`–`NB06`**: una pasada más profunda por las herramientas antes de empezar con Machine Learning — mecánica interna de arrays de NumPy y cálculo en ingeniería, una clase sistemática de Matplotlib, y SciPy para ingeniería naval.
- **`NB07`** (hoy): el flujo de trabajo real de Machine Learning — desde "qué es un modelo" hasta entrenar y evaluar dos modelos reales sobre un dataset real.

Estamos dentro del **Bloque 2 — IA: Machine Learning** de la hoja de ruta del curso (ver `NB01` para el plan completo de los cuatro bloques).

---

## 2. ¿Qué es el Machine Learning?

El [Machine Learning (ML)](https://en.wikipedia.org/wiki/Machine_learning) es un subconjunto de la [Inteligencia Artificial](https://en.wikipedia.org/wiki/Artificial_intelligence) que permite a los ordenadores **aprender y tomar decisiones a partir de datos**, en vez de estar programados explícitamente para cada regla. Se centra en algoritmos que `identifican patrones en los datos y usan esos patrones para hacer predicciones o tomar decisiones sobre datos nuevos, nunca vistos`.

Algunas propiedades a tener en cuenta antes de entrenar nada:
- **Impulsado por los datos**: la calidad del modelo está limitada por la calidad de los datos — "basura entra, basura sale".
- **Adaptativo**: los modelos se pueden reentrenar a medida que llegan nuevos datos; el rendimiento puede degradarse con el tiempo si las condiciones cambian (*model drift*, deriva del modelo).
- **El objetivo es generalizar**: un modelo que solo memoriza sus ejemplos de entrenamiento no es útil — debe funcionar bien con datos que nunca ha visto.

### Tipos de Machine Learning

| Tipo | ¿Datos etiquetados? | Objetivo | Algoritmos habituales | Ejemplo marítimo |
|---|:---:|---|---|---|
| **Aprendizaje supervisado** | Sí | Aprender una función que asocia entradas → salidas | Regresión Lineal/Logística, Árboles de Decisión, Random Forest, SVM | Predecir el consumo de combustible a partir de las condiciones del viaje |
| **Aprendizaje no supervisado** | No | Descubrir estructura/agrupaciones ocultas | K-Means, PCA, Clustering Jerárquico | Segmentar buques en perfiles operativos |
| **Aprendizaje por refuerzo** | No exactamente (señal de recompensa) | Aprender una política que maximiza la recompensa acumulada | Q-Learning, Policy Gradients, DQN | Optimización autónoma de ruta/velocidad |

La clase de hoy es enteramente de **aprendizaje supervisado**: los dos problemas que vamos a resolver (predecir el tipo de combustible, predecir el consumo de combustible) tienen un objetivo (target) conocido y etiquetado en el dataset.

> **Para saber más**: [Aprendizaje supervisado (Wikipedia)](https://en.wikipedia.org/wiki/Supervised_learning) · [Aprendizaje no supervisado (Wikipedia)](https://en.wikipedia.org/wiki/Unsupervised_learning) · [Aprendizaje por refuerzo (Wikipedia)](https://en.wikipedia.org/wiki/Reinforcement_learning).

### Conjuntos de entrenamiento, validación y test

- El **conjunto de entrenamiento** se usa para ajustar el modelo — los datos de los que realmente aprende el algoritmo.
- El **conjunto de validación** se usa para afinar decisiones *sobre* el modelo (qué algoritmo, qué hiperparámetros) sin tocar los datos de evaluación final.
- El **conjunto de test** se usa **una sola vez**, al final, para estimar cómo se comportará el modelo con datos genuinamente nuevos. Si consultas el conjunto de test mientras ajustas el modelo, tu evaluación se vuelve optimista y poco fiable — esto es una forma de **fuga de datos (data leakage)**.

Una proporción de división habitual es 70/15/15 o 60/20/20. Cuando no es práctico tener un conjunto de validación separado (por ejemplo, con pocos datos), se usa en su lugar la **validación cruzada k-fold** — la usaremos más adelante hoy.

### Sobreajuste, infraajuste y el equilibrio sesgo-varianza

- **Sobreajuste (overfitting)**: el modelo es demasiado complejo y aprende el ruido de los datos de entrenamiento, no solo su patrón — muy buen rendimiento en entrenamiento, pobre en test. *Varianza alta, sesgo bajo.*
- **Infraajuste (underfitting)**: el modelo es demasiado simple para capturar el patrón real — rendimiento pobre tanto en entrenamiento como en test. *Sesgo alto, varianza baja.*
- El **equilibrio sesgo-varianza** es el balance entre estos dos modos de fallo: a medida que aumenta la complejidad del modelo, el sesgo tiende a bajar pero la varianza tiende a subir. El objetivo es el nivel de complejidad que minimiza el error total sobre datos no vistos — normalmente se encuentra mediante validación cruzada.

| Complejidad del modelo | Sesgo | Varianza | Ejemplo |
|---|:---:|:---:|---|
| Demasiado simple | Alto | Baja | Regresión lineal sobre una relación fuertemente no lineal |
| Demasiado complejo | Bajo | Alta | Un árbol de decisión muy profundo sin límite de profundidad |
| Bien equilibrado | Moderado | Moderada | Random Forest con profundidad ajustada, o modelo lineal regularizado |

Lo veremos directamente hoy: Regresión Lineal (más simple, mayor sesgo) frente a Random Forest (más flexible, mayor riesgo de varianza si no se controla).

---

## 3. Métricas de evaluación

### Métricas de clasificación

Se usan cuando el objetivo (target) es categórico (p. ej., Diesel vs. HFO, spam vs. no spam).

- **Accuracy**: $\dfrac{TP + TN}{TP + TN + FP + FN}$ — corrección global; puede ser engañosa con datos desbalanceados.
- **Precisión (precision)**: $\dfrac{TP}{TP + FP}$ — de todo lo predicho como positivo, cuánto era realmente positivo.
- **Recall (sensibilidad)**: $\dfrac{TP}{TP + FN}$ — de todo lo que era realmente positivo, cuánto detectamos.
- **F1-score**: $2 \cdot \dfrac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — media armónica de precisión y recall, útil cuando hay desequilibrio de clases.

#### Ejemplo resuelto: detección de vertidos de petróleo (ilustrativo, calculado a mano)

Supongamos que un clasificador de imágenes de satélite se evalúa sobre 300 imágenes (un problema desbalanceado — los vertidos reales son raros) y produce esta matriz de confusión:

| | Predicho: vertido | Predicho: sin vertido |
|---|:---:|:---:|
| **Real: vertido** | TP = 42 | FN = 14 |
| **Real: sin vertido** | FP = 5 | TN = 239 |

$$
\text{Accuracy} = \frac{42 + 239}{300} \approx 93.7\% \qquad
\text{Precision} = \frac{42}{47} \approx 0.894 \qquad
\text{Recall} = \frac{42}{56} = 0.75 \qquad
F1 = 2 \cdot \frac{0.894 \cdot 0.75}{0.894 + 0.75} \approx 0.815
$$

`Incluso con un 93.7% de accuracy, el recall es solo del 75%: se pierde uno de cada cuatro vertidos reales`. Esta es exactamente la razón por la que **la accuracy sola es una métrica pobre para problemas desbalanceados y críticos para la seguridad** — una lección que aplicaremos de forma real más adelante hoy. En la práctica esto se calcula con `confusion_matrix()` y `classification_report()`, no a mano — verás ambas en un momento.

> **Para saber más**: [Matriz de confusión (Wikipedia)](https://en.wikipedia.org/wiki/Confusion_matrix) · [documentación de `sklearn.metrics.confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) · [documentación de `sklearn.metrics.classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html).

### Métricas de regresión

Se usan cuando el objetivo (target) es continuo (p. ej., consumo de combustible, temperatura).

| Métrica | Fórmula | Se interpreta como |
|---|---|---|
| **MAE** (Error Absoluto Medio) | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | Tamaño medio del error, mismas unidades que el objetivo, robusto a valores atípicos |
| **MSE** (Error Cuadrático Medio) | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ | Penaliza más los errores grandes |
| **RMSE** (Raíz del Error Cuadrático Medio) | $\sqrt{\text{MSE}}$ | Mismas unidades que el objetivo, más sensible a valores atípicos que el MAE |
| **R²** (Coeficiente de Determinación) | $1 - \dfrac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | Proporción de varianza explicada; 1.0 = perfecto, 0.0 = no mejor que predecir la media |

> **Regla general**: mira siempre el MAE/RMSE *en relación con la escala del objetivo*. Un RMSE de 500 L es excelente si el consumo de combustible típico es de 20.000 L, y terrible si es de 600 L. Tendremos esto en cuenta al evaluar nuestro propio modelo más adelante — la columna `fuel_consumption` del dataset promedia **~4.844 L** por viaje, con valores que van desde unos 238 hasta 24.650 L.

> **Para saber más**: [Coeficiente de determinación (Wikipedia)](https://en.wikipedia.org/wiki/Coefficient_of_determination) · [documentación de `sklearn.metrics.r2_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) · [documentación de `sklearn.metrics.mean_absolute_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html).

---

## 4. Carga y exploración del dataset real

Ahora pasamos por completo a datos reales: **[Ship Fuel Consumption and CO2 Emissions Analysis](https://www.kaggle.com/datasets/jeleeladekunlefijabi/ship-fuel-consumption-and-co2-emissions-analysis)**, publicado en Kaggle, que cubre 120 buques en rutas fluviales de Nigeria durante 12 meses. Ya está reflejado en este repositorio, así que lo cargamos igual que cargamos `Naval_Dataset.csv` en `NB02` — no hace falta clave de API de Kaggle para esta clase.

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
print(fuel.shape)
fuel.head()

Cada fila es un viaje de un buque en un mes. Columnas:

| Columna | Significado |
|---|---|
| `ship_id` | Identificador del buque (120 buques únicos) |
| `ship_type` | Oil Service Boat, Tanker Ship, Surfer Boat, o Fishing Trawler |
| `route_id` | Una de 4 rutas fluviales de Nigeria |
| `month` | Mes del viaje (ene-dic) |
| `distance` | Distancia recorrida (millas náuticas) |
| `fuel_type` | Diesel o HFO (Heavy Fuel Oil, fuel oil pesado) |
| `fuel_consumption` | Combustible quemado en el viaje (litros) — **nuestro objetivo de regresión** |
| `CO2_emissions` | CO₂ emitido en el viaje (kg) |
| `weather_conditions` | Calm, Moderate, o Stormy |
| `engine_efficiency` | Eficiencia del motor (%) |

In [ ]:
fuel.info()

Ahora los rangos numéricos y las estadísticas resumen:

In [ ]:
fuel.describe()

Y las categorías que toma cada columna:

In [ ]:
for col in ["ship_type", "route_id", "fuel_type", "weather_conditions"]:
    print(fuel[col].value_counts())
    print()

### Preguntas exploratorias rápidas

Antes de modelar nada, siempre hay que buscar patrones y hacer una comprobación de sentido de los datos con `groupby()` — exactamente como practicamos en `NB02`.

In [ ]:
fuel.groupby("ship_type")["fuel_consumption"].mean().sort_values(ascending=False)

La misma pregunta, agrupando por clima en su lugar:

In [ ]:
fuel.groupby("weather_conditions")["fuel_consumption"].mean().sort_values(ascending=False)

Visualiza esa misma comparación como un boxplot:

In [ ]:
import matplotlib.pyplot as plt

fuel.boxplot(column="fuel_consumption", by="weather_conditions", figsize=(6, 4))
plt.title("Fuel consumption by weather condition")
plt.suptitle("")
plt.xlabel("Weather condition")
plt.ylabel("Fuel consumption (L)")
plt.show()

### Una trampa de fuga de datos, encontrada de verdad

Vamos a comprobar cómo se correlacionan entre sí las columnas numéricas.

In [ ]:
numeric_cols = ["distance", "fuel_consumption", "CO2_emissions", "engine_efficiency"]
fuel[numeric_cols].corr()

Representa esa matriz de correlación para que la fuga salte a la vista:

In [ ]:
plt.figure(figsize=(5.5, 5))
plt.imshow(fuel[numeric_cols].corr(), cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

Deberías ver una correlación de aproximadamente **0.997** entre `fuel_consumption` y `CO2_emissions`. `Eso tiene sentido físico — el CO₂ emitido es esencialmente una función directa y casi lineal del combustible quemado`. También significa: **si usáramos `CO2_emissions` como característica de entrada para predecir `fuel_consumption`, el modelo en realidad no estaría aprendiendo nada — simplemente estaría leyendo un valor casi equivalente.** Esto es **fuga de datos (data leakage)**: una característica que (directa o indirectamente) contiene la respuesta.

Excluiremos `CO2_emissions` de las características de regresión más abajo por exactamente esta razón.

> **Para saber más**: [Fuga de datos en machine learning (Wikipedia)](https://en.wikipedia.org/wiki/Leakage_%28machine_learning%29).

---

## 5. Ingeniería de características y preprocesado

Los datos reales rara vez están listos para un modelo tal cual. Dos transformaciones que necesitamos hoy:

| Técnica | Propósito | Aquí, aplicada a |
|---|---|---|
| **Encoding (codificación)** | Los modelos de ML necesitan números, no categorías de texto | `ship_type`, `route_id`, `fuel_type`, `weather_conditions` → columnas one-hot vía `pd.get_dummies()` |
| **Escalado (scaling)** | Pone las características numéricas en rangos comparables (ayuda a modelos lineales/basados en distancia) | `distance`, `engine_efficiency` → `StandardScaler` |
| **Eliminación de fuga** | Evita que una característica codifique la respuesta | Eliminar `CO2_emissions` de las entradas de regresión (ver más arriba) |

`fuel.info()` mostró arriba **sin valores perdidos** en este dataset, así que hoy podemos saltarnos la imputación — pero en un pipeline real basado en sensores (p. ej., telemetría a bordo), el tratamiento de valores perdidos/duplicados (`dropna()`, `fillna()`, `drop_duplicates()`) normalmente iría primero.

---

## 6. Clasificación en la práctica: predecir el tipo de combustible

**Tarea**: dado el tipo de buque, la ruta, la distancia, el clima y la eficiencia del motor de un viaje, predecir si usó **Diesel** o **HFO**. Este es un problema genuino de clasificación binaria — el tipo de combustible *no* está completamente determinado solo por el tipo de buque (solo `Surfer Boat` en esta flota usa exclusivamente Diesel; los otros tres tipos de buque mezclan ambos combustibles), así que hay un patrón real que el modelo puede aprender, pero no es trivial.

> **Para saber más**: [documentación de `sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

clf_features = ["ship_type", "route_id", "distance", "weather_conditions", "engine_efficiency"]

X_clf = pd.get_dummies(fuel[clf_features], columns=["ship_type", "route_id", "weather_conditions"], drop_first=True)
y_clf = (fuel["fuel_type"] == "HFO").astype(int)  # 1 = HFO, 0 = Diesel

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler_clf = StandardScaler()
X_clf_train_scaled = scaler_clf.fit_transform(X_clf_train)
X_clf_test_scaled = scaler_clf.transform(X_clf_test)

print("Features used:", list(X_clf.columns))
print("Train size:", X_clf_train.shape, " Test size:", X_clf_test.shape)

> **Nota**: `stratify=y_clf` mantiene la misma proporción de Diesel/HFO tanto en la división de entrenamiento como en la de test — importante siempre que las clases no estén perfectamente balanceadas (aquí es aproximadamente 62%/38%).

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_clf_train_scaled, y_clf_train)
y_clf_pred = clf.predict(X_clf_test_scaled)

Ahora evalúa el clasificador: matriz de confusión e informe completo de métricas:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(y_clf_test, y_clf_pred)
ConfusionMatrixDisplay(cm, display_labels=["Diesel", "HFO"]).plot(cmap="Blues")
plt.title("Fuel type classification — confusion matrix")
plt.show()

print(classification_report(y_clf_test, y_clf_pred, target_names=["Diesel", "HFO"]))

**Pruébalo tú mismo**: calcula la accuracy real de la línea base "predecir siempre la clase mayoritaria" sobre este mismo conjunto de test, y compárala directamente con la accuracy real del clasificador de arriba — ¿se está ganando el modelo su complejidad?

In [ ]:
from sklearn.metrics import accuracy_score

majority_class_accuracy = max(y_clf_test.mean(), 1 - y_clf_test.mean())
model_accuracy = accuracy_score(y_clf_test, y_clf_pred)

print(f"Always-predict-majority-class baseline accuracy: {majority_class_accuracy:.1%}")
print(f"Logistic Regression accuracy: {model_accuracy:.1%}")
print(f"Improvement over baseline: {(model_accuracy - majority_class_accuracy):.1%} points")


**Interpreta tu propio resultado** (variará ligeramente entre ejecuciones, ya que tanto `LogisticRegression` como la división tienen algo de aleatoriedad controlada por `random_state`, pero debería ser en general estable):
- ¿Está la accuracy claramente por encima de la línea base de "predecir siempre la clase mayoritaria" (~62%)?
- ¿Son la precisión y el recall de la clase minoritaria (la que sea en tu ejecución) notablemente más bajos que los de la clase mayoritaria? ¿Por qué podría ser eso, dado lo que hablamos sobre el desequilibrio?
- ¿Qué características de ruta/tipo de buque investigarías a continuación si el recall de HFO fuera bajo?

---

## 7. Regresión en la práctica: predecir el consumo de combustible

**Tarea**: predecir `fuel_consumption` (litros) a partir de las condiciones del viaje — excluyendo `CO2_emissions` por la razón de fuga de datos establecida arriba, y excluyendo `ship_id` (solo un identificador, no una característica real). `Vamos a entrenar dos modelos y compararlos, exactamente el equilibrio sesgo-varianza discutido en la Parte 2`: una **Regresión Lineal** simple y un **Random Forest** más flexible.

> **Para saber más**: [documentación de `sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [documentación de `sklearn.ensemble.RandomForestRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html).

In [ ]:
reg_features = ["ship_type", "distance", "fuel_type", "weather_conditions", "engine_efficiency"]

X_reg = pd.get_dummies(fuel[reg_features], columns=["ship_type", "fuel_type", "weather_conditions"], drop_first=True)
y_reg = fuel["fuel_consumption"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_reg_train_scaled = scaler_reg.fit_transform(X_reg_train)
X_reg_test_scaled = scaler_reg.transform(X_reg_test)

print("Features used:", list(X_reg.columns))

Entrena y compara ambos modelos sobre la misma tarea de predicción real:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_reg_train_scaled, y_reg_train)
    y_reg_pred = model.predict(X_reg_test_scaled)
    results.append({
        "Model": name,
        "MAE (L)": mean_absolute_error(y_reg_test, y_reg_pred),
        "RMSE (L)": mean_squared_error(y_reg_test, y_reg_pred) ** 0.5,
        "R2": r2_score(y_reg_test, y_reg_pred),
    })

results_df = pd.DataFrame(results)
results_df

Recuerda que `fuel_consumption` en el dataset va aproximadamente de 238 a 24.650 L, promediando ~4.844 L. Usa esa escala para juzgar si el MAE/RMSE de arriba son realmente buenos — una métrica solo significa algo en relación con el propio rango del objetivo.

In [ ]:
best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X_reg_train_scaled, y_reg_train)
y_reg_pred = best_model.predict(X_reg_test_scaled)

plt.figure(figsize=(6, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.5)
lims = [y_reg.min(), y_reg.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual fuel consumption (L)")
plt.ylabel("Predicted fuel consumption (L)")
plt.title("Random Forest — predicted vs. actual")
plt.legend()
plt.show()

Los puntos cerca de la diagonal roja son viajes que el modelo predijo bien; los puntos lejos de ella son los mayores fallos del modelo. Si tienes tiempo, prueba a filtrar `fuel.loc[y_reg_test.index]` por las filas de mayor error — ¿hay un patrón (un tipo de buque, ruta o condición climática concreta) entre las peores predicciones?

**Pruébalo tú mismo**: así es como se usaría el modelo en la práctica realmente — dadas las condiciones de un viaje *nuevo*, nunca visto (que no está en el dataset en absoluto), predecir su consumo de combustible. Construye un viaje hipotético (un Tanker Ship, 250 mn, HFO, clima Stormy, 75% de eficiencia del motor) y obtén una predicción real del modelo entrenado. (Cuidado: las columnas one-hot deben coincidir exactamente con las columnas de entrenamiento — usa `.reindex(columns=X_reg.columns, fill_value=0)` después de codificar.)

In [ ]:
hypothetical_voyage = pd.DataFrame([{
    "ship_type": "Tanker Ship",
    "distance": 250.0,
    "fuel_type": "HFO",
    "weather_conditions": "Stormy",
    "engine_efficiency": 75.0,
}])

hypothetical_encoded = pd.get_dummies(hypothetical_voyage, columns=["ship_type", "fuel_type", "weather_conditions"])
hypothetical_encoded = hypothetical_encoded.reindex(columns=X_reg.columns, fill_value=0)   # match training columns exactly
hypothetical_scaled = scaler_reg.transform(hypothetical_encoded)

predicted_fuel = best_model.predict(hypothetical_scaled)
print(f"Predicted fuel consumption for this hypothetical voyage: {predicted_fuel[0]:.0f} L")


---

## 8. Validación cruzada sobre el modelo real

`Una única división train/test puede tener suerte o mala suerte, especialmente con un dataset de este tamaño`. La **validación cruzada k-fold** divide los datos en *k* partes, entrena con *k−1* de ellas, y evalúa con la restante — repitiendo *k* veces para que cada fila se use exactamente una vez para evaluar — y después informa de la media y la dispersión de la puntuación entre folds.

| Técnica | Cuándo usarla | Notas |
|---|---|---|
| **K-Fold (k=5 o 10)** | Uso general, la mayoría de tareas de regresión/clasificación | Buen equilibrio sesgo/varianza |
| **Stratified K-Fold** | Clasificación con desequilibrio de clases | Conserva las proporciones de clase en cada fold — es lo que usaríamos para el clasificador de tipo de combustible |
| **Leave-One-Out (LOOCV)** | Datasets muy pequeños | Sesgo bajo, pero lento y con varianza alta; k = n |

> **Para saber más**: [documentación de `sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) · [documentación de `sklearn.model_selection.KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html).

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42),
    X_reg, y_reg, cv=kf, scoring="r2"
)

print("R2 per fold:", cv_scores.round(3))
print(f"Mean R2: {cv_scores.mean():.3f}  (+/- {cv_scores.std():.3f})")

**Pruébalo tú mismo**: valida también con validación cruzada el modelo más simple de Regresión Lineal, usando exactamente los mismos folds (`cv=kf`) — ¿se mantiene la ventaja de Random Forest de la división única de la Parte 7 en los 5 folds, o se reduce?

In [ ]:
cv_scores_linear = cross_val_score(LinearRegression(), X_reg, y_reg, cv=kf, scoring="r2")

print("Linear Regression -- R2 per fold:", cv_scores_linear.round(3))
print(f"Linear Regression -- Mean R2: {cv_scores_linear.mean():.3f}  (+/- {cv_scores_linear.std():.3f})")
print(f"\nRandom Forest -- Mean R2: {cv_scores.mean():.3f}  (+/- {cv_scores.std():.3f})")


Si la R² media validada de forma cruzada está cerca de la R² de la división única de la Parte 7, es buena señal — la división única no fue un resultado afortunado/desafortunado. Si las puntuaciones de los folds varían mucho, el rendimiento del modelo es menos estable de lo que sugería una única división, lo cual importa si este modelo se usara alguna vez de forma operativa.

---

## 9. Aplicaciones del Machine Learning en ingeniería naval y oceánica (panorama general)

El trabajo práctico de hoy cubrió regresión y clasificación *tabular*. `El ML también se aplica a otros tipos de datos relevantes para este curso` — volveremos a la visión artificial y a los problemas de series temporales/secuencias en el próximo bloque de **Deep Learning**.

| Dominio | Ejemplo marítimo/oceánico | Enfoque típico |
|---|---|---|
| Visión artificial | Detección de buques/obstáculos en imágenes de satélite o dron; inspección submarina con ROV | CNNs (bloque de Deep Learning) |
| Series temporales / secuencias | Previsión de altura de ola o estado del mar; mantenimiento predictivo a partir de sensores de vibración | RNNs, modelos temporales (bloque de Deep Learning) |
| Clustering (no supervisado) | Segmentar buques o viajes en perfiles operativos sin etiquetas | K-Means, PCA |
| Procesamiento del lenguaje natural | Clasificar partes de mantenimiento, informes de incidentes | Clasificación de texto |
| Detección de anomalías | Señalar lecturas anómalas del motor antes de un fallo | Isolation Forest, autoencoders |

---

## Resumen de la clase

- El ML se divide en aprendizaje supervisado, no supervisado y por refuerzo; hoy fue enteramente supervisado.
- La separación entrenamiento/validación/test — y evitar la fuga de datos — es lo que hace fiable una evaluación. Encontramos una trampa de fuga real (`CO2_emissions`) en el propio dataset de hoy.
- Las métricas de clasificación (accuracy, precisión, recall, F1) y las de regresión (MAE, RMSE, R²) responden a preguntas distintas; hay que leerlas siempre en relación con el problema (desequilibrio de clases, escala del objetivo).
- Entrenamos y evaluamos dos modelos reales — un clasificador de Regresión Logística y un par de regresión Regresión Lineal/Random Forest — sobre un dataset marítimo real y publicado, no datos simulados.
- La validación cruzada k-fold da una estimación de rendimiento más robusta que una única división train/test.

## Para la próxima clase (NB08)

Profundizaremos en algoritmos supervisados concretos (árboles de decisión, ensembles, máquinas de vector soporte) y empezaremos a mirar el aprendizaje no supervisado (clustering), cubriendo un hueco identificado frente a la guía docente del curso.

## Tarea / Ideas de práctica

1. Añade `route_id` y/o `month` como características codificadas adicionales al modelo de regresión de la Parte 7 — ¿mejora la R²? ¿Merece la pena la mejora frente a la complejidad añadida?
2. Repite la tarea de clasificación de la Parte 6, pero prediciendo `weather_conditions` (3 clases: Calm/Moderate/Stormy) en vez del tipo de combustible. ¿Qué métricas de la Parte 3 siguen aplicando directamente, y cuáles hay que ajustar para más de dos clases?
3. Explica con tus propias palabras por qué era necesario excluir `CO2_emissions` de las características de regresión — y da otro ejemplo (naval o no) de una característica que podría filtrar la respuesta.
4. Compara los errores de Regresión Lineal y Random Forest de la Parte 7 por tipo de buque (pista: agrupa con `groupby` los errores del conjunto de test por `ship_type`) — ¿es un modelo mucho mejor para un tipo de buque concreto? Propón una razón.
5. Cambia `cv=kf` en la Parte 8 por `cv=10` — ¿cómo cambian la media y la desviación típica de la R², y por qué esperarías eso?

> ***Como siempre: plantea cada pregunta pensando en lo que realmente querría saber de este modelo un ingeniero naval o un operador de flota.***

> Para un tratamiento más profundo, a nivel de libro de texto, de regresión, clasificación y validación cruzada en un solo lugar, ver el libro de texto gratuito [*An Introduction to Statistical Learning*](https://www.statlearning.com/) (James, Witten, Hastie & Tibshirani).